In [ ]:
!pip uninstall -y -q pyyaml
!pip install -q --no-cache-dir "pyyaml==6.0.2" "huggingface-hub>=0.34,<1.0" "transformers>=4.45,<5" "sentence-transformers>=3.0,<4" "accelerate>=1.0,<2" "xgboost>=2.0,<4"

In [1]:
DEBUG = False
EVAL_SIZE = 100 if DEBUG else 1000

RETRIEVAL_DATASET = "/kaggle/input/datasets/keyaaness/cost-aware-adaptive-rag-retrieval-v1/retrieval_artifacts"
BASELINE_DATASET = "/kaggle/input/datasets/keyaaness/cost-aware-adaptive-rag-fixed-baseline-v2"
CONTROLLER_DATASET = "/kaggle/input/datasets/keyaaness/cost-aware-adaptive-rag-controller-v2-final"

OUTPUT_DIR = "/kaggle/working/final_results_v2"

EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
GENERATOR_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

MAX_NEW_TOKENS = 64
SEED = 42

!pip install -q sentence-transformers transformers accelerate pandas numpy pyarrow faiss-cpu xgboost matplotlib

import os
import re
import json
import pickle
import random
import time

import numpy as np
import pandas as pd
import torch
import faiss
import matplotlib.pyplot as plt

from pathlib import Path
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

corpus = pd.read_parquet(
    os.path.join(RETRIEVAL_DATASET, "corpus.parquet")
)

final_test = pd.read_parquet(
    os.path.join(RETRIEVAL_DATASET, "final_test.parquet")
)

faiss_index = faiss.read_index(
    os.path.join(RETRIEVAL_DATASET, "faiss.index")
)

fixed_results = pd.read_csv(
    os.path.join(
        BASELINE_DATASET,
        "fixed_rag_results.csv"
    )
)

with open(
    os.path.join(
        CONTROLLER_DATASET,
        "controller.pkl"
    ),
    "rb"
) as f:
    controller = pickle.load(f)

with open(
    os.path.join(
        CONTROLLER_DATASET,
        "scaler.pkl"
    ),
    "rb"
) as f:
    scaler = pickle.load(f)

with open(
    os.path.join(
        CONTROLLER_DATASET,
        "feature_names.json"
    )
) as f:
    feature_names = json.load(f)

with open(
    os.path.join(
        CONTROLLER_DATASET,
        "selected_expansion_rate.json"
    )
) as f:
    controller_config = json.load(f)

expansion_rate = float(
    controller_config["expansion_rate"]
)

eval_df = final_test.sample(
    n=EVAL_SIZE,
    random_state=SEED
).reset_index(drop=True)

eval_df["id"] = eval_df["id"].astype(str)
fixed_results["id"] = fixed_results["id"].astype(str)

fixed_results = fixed_results[
    fixed_results["id"].isin(
        set(eval_df["id"])
    )
].copy()

if set(fixed_results["id"]) != set(eval_df["id"]):
    raise ValueError(
        "Fixed baseline and adaptive evaluation IDs do not match."
    )

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL,
    device=device
)

generator_tokenizer = AutoTokenizer.from_pretrained(
    GENERATOR_MODEL,
    use_fast=True
)

dtype = (
    torch.float16
    if torch.cuda.is_available()
    else torch.float32
)

generator = AutoModelForCausalLM.from_pretrained(
    GENERATOR_MODEL,
    torch_dtype=dtype
)

generator.to(device)
generator.eval()

if generator_tokenizer.pad_token_id is None:
    generator_tokenizer.pad_token = generator_tokenizer.eos_token

generator.generation_config.do_sample = False
generator.generation_config.temperature = None
generator.generation_config.top_p = None
generator.generation_config.top_k = None

print("Device:", device)
print("Evaluation examples:", len(eval_df))
print("Frozen expansion rate:", expansion_rate)

`torch_dtype` is deprecated! Use `dtype` instead!


Device: cuda
Evaluation examples: 1000
Frozen expansion rate: 0.1


In [2]:
def dense_retrieve(query, k):
    embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    ).astype(np.float32)

    scores, indices = faiss_index.search(
        embedding,
        k
    )

    documents = []

    for score, idx in zip(
        scores[0],
        indices[0]
    ):
        row = corpus.iloc[int(idx)]

        documents.append({
            "doc_id": int(row["doc_id"]),
            "title": str(row["title"]),
            "text": str(row["text"]),
            "score": float(score)
        })

    return documents


def build_context(documents):
    return "\n\n".join(
        f"[Document {i}]\n"
        f"Title: {document['title']}\n"
        f"{document['text']}"
        for i, document in enumerate(
            documents,
            1
        )
    )


def make_router_features(
    query,
    documents
):
    scores = np.array(
        [
            d["score"]
            for d in documents[:3]
        ],
        dtype=np.float32
    )

    values = {
        "query_tokens": len(
            re.findall(
                r"[A-Za-z0-9]+",
                str(query).lower()
            )
        ),
        "query_chars": len(str(query)),
        "top1": float(scores[0]),
        "top2": float(scores[1]),
        "top3": float(scores[2]),
        "gap12": float(
            scores[0] - scores[1]
        ),
        "gap23": float(
            scores[1] - scores[2]
        ),
        "gap13": float(
            scores[0] - scores[2]
        ),
        "mean3": float(
            scores.mean()
        ),
        "std3": float(
            scores.std()
        ),
        "top1_mean3_ratio": float(
            scores[0] /
            (abs(scores.mean()) + 1e-8)
        )
    }

    return pd.DataFrame(
        [[
            values[name]
            for name in feature_names
        ]],
        columns=feature_names
    ).astype(np.float32)


def generate_answer(
    question,
    context
):
    messages = [
        {
            "role": "system",
            "content": (
                "Answer using only the provided context. "
                "Return only the concise answer."
            )
        },
        {
            "role": "user",
            "content": (
                f"Context:\n{context}\n\n"
                f"Question: {question}\n"
                f"Answer:"
            )
        }
    ]

    prompt = generator_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = generator_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    prompt_length = inputs["input_ids"].shape[1]

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start = time.perf_counter()

    with torch.inference_mode():
        output = generator.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1,
            pad_token_id=generator_tokenizer.pad_token_id,
            eos_token_id=generator_tokenizer.eos_token_id
        )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start

    answer = generator_tokenizer.decode(
        output[0][prompt_length:],
        skip_special_tokens=True
    ).strip()

    return answer, elapsed


def normalize_answer(text):
    text = str(text).lower()
    text = re.sub(
        r"\b(a|an|the)\b",
        " ",
        text
    )
    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text
    )
    text = re.sub(
        r"\s+",
        " ",
        text
    )
    return text.strip()


def exact_match(
    prediction,
    reference
):
    return float(
        normalize_answer(prediction)
        ==
        normalize_answer(reference)
    )


def token_f1(
    prediction,
    reference
):
    p = normalize_answer(
        prediction
    ).split()

    r = normalize_answer(
        reference
    ).split()

    if not p and not r:
        return 1.0

    if not p or not r:
        return 0.0

    pc = {}
    rc = {}

    for token in p:
        pc[token] = pc.get(token, 0) + 1

    for token in r:
        rc[token] = rc.get(token, 0) + 1

    overlap = sum(
        min(
            pc[token],
            rc.get(token, 0)
        )
        for token in pc
    )

    if overlap == 0:
        return 0.0

    precision = overlap / len(p)
    recall = overlap / len(r)

    return (
        2 * precision * recall
        / (precision + recall)
    )


def count_tokens(text):
    return len(
        generator_tokenizer.encode(
            text,
            add_special_tokens=False
        )
    )

In [3]:
router_scores = []
top3_documents = []
initial_retrieval_times = []

for _, row in tqdm(
    eval_df.iterrows(),
    total=len(eval_df),
    desc="Scoring test queries"
):
    query = str(row["question"])

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start = time.perf_counter()

    documents = dense_retrieve(
        query,
        3
    )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    retrieval_time = (
        time.perf_counter()
        - start
    )

    features = make_router_features(
        query,
        documents
    )

    probability = float(
        controller.predict_proba(
            scaler.transform(features)
        )[0, 1]
    )

    top3_documents.append(
        documents
    )

    router_scores.append(
        probability
    )

    initial_retrieval_times.append(
        retrieval_time
    )

n_expand = max(
    1,
    int(
        np.ceil(
            expansion_rate
            * len(eval_df)
        )
    )
)

rank_order = np.argsort(
    np.asarray(router_scores)
)[::-1]

expand_mask = np.zeros(
    len(eval_df),
    dtype=bool
)

expand_mask[
    rank_order[:n_expand]
] = True

results = []

for i, (_, row) in enumerate(
    tqdm(
        eval_df.iterrows(),
        total=len(eval_df),
        desc="Adaptive RAG evaluation"
    )
):
    query = str(row["question"])
    gold = str(row["answer"])

    documents = top3_documents[i]

    additional_retrieval_time = 0.0

    if expand_mask[i]:

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        start = time.perf_counter()

        documents = dense_retrieve(
            query,
            5
        )

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        additional_retrieval_time = (
            time.perf_counter()
            - start
        )

        route = "top5"

    else:
        route = "top3"

    context = build_context(
        documents
    )

    prediction, generation_time = generate_answer(
        query,
        context
    )

    retrieval_time = (
        initial_retrieval_times[i]
        + additional_retrieval_time
    )

    results.append({
        "id": str(row["id"]),
        "question": query,
        "gold_answer": gold,
        "prediction": prediction,
        "route": route,
        "controller_score": router_scores[i],
        "retrieved_k": len(documents),
        "context_tokens": count_tokens(
            context
        ),
        "generation_tokens": len(
            generator_tokenizer.encode(
                prediction,
                add_special_tokens=False
            )
        ),
        "retrieval_time_ms": retrieval_time * 1000,
        "generation_time_ms": generation_time * 1000,
        "total_time_ms": (
            retrieval_time
            + generation_time
        ) * 1000,
        "exact_match": exact_match(
            prediction,
            gold
        ),
        "f1": token_f1(
            prediction,
            gold
        )
    })

adaptive_df = pd.DataFrame(
    results
)

print(
    "Top-3 queries:",
    int((adaptive_df["route"] == "top3").sum())
)

print(
    "Top-5 queries:",
    int((adaptive_df["route"] == "top5").sum())
)

Scoring test queries:   0%|          | 0/1000 [00:00<?, ?it/s]

Adaptive RAG evaluation:   0%|          | 0/1000 [00:00<?, ?it/s]

Top-3 queries: 900
Top-5 queries: 100


In [4]:
fixed = fixed_results[
    [
        "id",
        "prediction",
        "context_tokens",
        "generation_tokens",
        "retrieval_time_ms",
        "generation_time_ms",
        "total_time_ms",
        "exact_match",
        "f1"
    ]
].rename(
    columns={
        "prediction": "fixed_prediction",
        "context_tokens": "fixed_context_tokens",
        "generation_tokens": "fixed_generation_tokens",
        "retrieval_time_ms": "fixed_retrieval_time_ms",
        "generation_time_ms": "fixed_generation_time_ms",
        "total_time_ms": "fixed_total_time_ms",
        "exact_match": "fixed_exact_match",
        "f1": "fixed_f1"
    }
)

comparison = adaptive_df.merge(
    fixed,
    on="id",
    how="inner",
    validate="one_to_one"
)

if len(comparison) != len(adaptive_df):
    raise ValueError(
        "Fixed and adaptive results do not match."
    )

fixed_em = float(
    comparison["fixed_exact_match"].mean()
)

adaptive_em = float(
    comparison["exact_match"].mean()
)

fixed_f1 = float(
    comparison["fixed_f1"].mean()
)

adaptive_f1 = float(
    comparison["f1"].mean()
)

fixed_context = float(
    comparison["fixed_context_tokens"].mean()
)

adaptive_context = float(
    comparison["context_tokens"].mean()
)

fixed_generation = float(
    comparison["fixed_generation_tokens"].mean()
)

adaptive_generation = float(
    comparison["generation_tokens"].mean()
)

fixed_latency = float(
    comparison["fixed_total_time_ms"].mean()
)

adaptive_latency = float(
    comparison["total_time_ms"].mean()
)

fixed_median_latency = float(
    comparison["fixed_total_time_ms"].median()
)

adaptive_median_latency = float(
    comparison["total_time_ms"].median()
)

fixed_p95_latency = float(
    comparison["fixed_total_time_ms"].quantile(0.95)
)

adaptive_p95_latency = float(
    comparison["total_time_ms"].quantile(0.95)
)

top3_rate = float(
    np.mean(
        comparison["route"] == "top3"
    )
)

top5_rate = float(
    np.mean(
        comparison["route"] == "top5"
    )
)

average_k = float(
    comparison["retrieved_k"].mean()
)

context_reduction = (
    1.0
    - adaptive_context / fixed_context
)

latency_reduction = (
    1.0
    - adaptive_latency / fixed_latency
)

summary = pd.DataFrame({
    "Metric": [
        "Exact Match",
        "F1",
        "Avg Context Tokens",
        "Avg Generated Tokens",
        "Avg Retrieval Depth",
        "Avg Total Latency (ms)",
        "Median Total Latency (ms)",
        "P95 Total Latency (ms)",
        "Top-3 Route Rate",
        "Top-5 Expansion Rate",
        "Context Token Reduction",
        "Latency Reduction"
    ],
    "Fixed Dense@5": [
        fixed_em,
        fixed_f1,
        fixed_context,
        fixed_generation,
        5.0,
        fixed_latency,
        fixed_median_latency,
        fixed_p95_latency,
        0.0,
        1.0,
        0.0,
        0.0
    ],
    "Adaptive 3→5": [
        adaptive_em,
        adaptive_f1,
        adaptive_context,
        adaptive_generation,
        average_k,
        adaptive_latency,
        adaptive_median_latency,
        adaptive_p95_latency,
        top3_rate,
        top5_rate,
        context_reduction,
        latency_reduction
    ]
})

output_path = Path(
    OUTPUT_DIR
)

output_path.mkdir(
    parents=True,
    exist_ok=True
)

(output_path / "figures").mkdir(
    exist_ok=True
)

adaptive_df.to_csv(
    output_path /
    "adaptive_rag_results.csv",
    index=False
)

comparison.to_csv(
    output_path /
    "fixed_vs_adaptive_results.csv",
    index=False
)

summary.to_csv(
    output_path /
    "comparison_summary.csv",
    index=False
)

routing_summary = pd.DataFrame({
    "route": [
        "top3",
        "top5"
    ],
    "queries": [
        int(
            (comparison["route"] == "top3").sum()
        ),
        int(
            (comparison["route"] == "top5").sum()
        )
    ],
    "percentage": [
        top3_rate * 100,
        top5_rate * 100
    ]
})

routing_summary.to_csv(
    output_path /
    "routing_summary.csv",
    index=False
)

qualitative = comparison[
    [
        "id",
        "question",
        "gold_answer",
        "fixed_prediction",
        "prediction",
        "route",
        "controller_score",
        "fixed_f1",
        "f1",
        "fixed_context_tokens",
        "context_tokens"
    ]
].copy()

qualitative["f1_change"] = (
    qualitative["f1"]
    - qualitative["fixed_f1"]
)

qualitative.sort_values(
    "f1_change"
).head(30).to_csv(
    output_path /
    "qualitative_examples.csv",
    index=False
)

metrics = {
    "evaluation_examples": int(
        len(comparison)
    ),
    "controller_expansion_rate": expansion_rate,
    "fixed_rag": {
        "exact_match": fixed_em,
        "f1": fixed_f1,
        "avg_context_tokens": fixed_context,
        "avg_generated_tokens": fixed_generation,
        "avg_retrieval_depth": 5.0,
        "avg_total_latency_ms": fixed_latency,
        "median_total_latency_ms": fixed_median_latency,
        "p95_total_latency_ms": fixed_p95_latency
    },
    "adaptive_rag": {
        "exact_match": adaptive_em,
        "f1": adaptive_f1,
        "avg_context_tokens": adaptive_context,
        "avg_generated_tokens": adaptive_generation,
        "avg_retrieval_depth": average_k,
        "avg_total_latency_ms": adaptive_latency,
        "median_total_latency_ms": adaptive_median_latency,
        "p95_total_latency_ms": adaptive_p95_latency,
        "top3_rate": top3_rate,
        "top5_rate": top5_rate
    },
    "reductions": {
        "context_token_reduction": context_reduction,
        "latency_reduction": latency_reduction
    }
}

with open(
    output_path /
    "metrics.json",
    "w"
) as f:
    json.dump(
        metrics,
        f,
        indent=2
    )

fig, axes = plt.subplots(
    1,
    2,
    figsize=(10, 4)
)

axes[0].bar(
    ["Fixed", "Adaptive"],
    [fixed_em, adaptive_em]
)

axes[0].set_title(
    "Exact Match"
)

axes[1].bar(
    ["Fixed", "Adaptive"],
    [fixed_f1, adaptive_f1]
)

axes[1].set_title(
    "F1"
)

plt.tight_layout()

plt.savefig(
    output_path /
    "figures" /
    "quality_comparison.png",
    dpi=200,
    bbox_inches="tight"
)

plt.close()

fig, axes = plt.subplots(
    1,
    2,
    figsize=(10, 4)
)

axes[0].bar(
    ["Fixed", "Adaptive"],
    [fixed_context, adaptive_context]
)

axes[0].set_title(
    "Average Context Tokens"
)

axes[1].bar(
    ["Fixed", "Adaptive"],
    [fixed_latency, adaptive_latency]
)

axes[1].set_title(
    "Average Total Latency (ms)"
)

plt.tight_layout()

plt.savefig(
    output_path /
    "figures" /
    "context_cost_comparison.png",
    dpi=200,
    bbox_inches="tight"
)

plt.close()

plt.figure(
    figsize=(7, 4)
)

plt.bar(
    routing_summary["route"],
    routing_summary["percentage"]
)

plt.ylabel(
    "Queries (%)"
)

plt.title(
    "Adaptive Routing Distribution"
)

plt.tight_layout()

plt.savefig(
    output_path /
    "figures" /
    "routing_distribution.png",
    dpi=200,
    bbox_inches="tight"
)

plt.close()

plt.figure(
    figsize=(7, 5)
)

plt.scatter(
    comparison["fixed_context_tokens"],
    comparison["fixed_f1"],
    alpha=0.35,
    label="Fixed Dense@5"
)

plt.scatter(
    comparison["context_tokens"],
    comparison["f1"],
    alpha=0.35,
    label="Adaptive 3→5"
)

plt.xlabel(
    "Context Tokens"
)

plt.ylabel(
    "F1"
)

plt.title(
    "Answer Quality vs Context Cost"
)

plt.legend()

plt.tight_layout()

plt.savefig(
    output_path /
    "figures" /
    "quality_vs_context_cost.png",
    dpi=200,
    bbox_inches="tight"
)

plt.close()

print("=" * 70)
print("FINAL ADAPTIVE RAG RESULTS")
print("=" * 70)
print(
    summary.to_string(
        index=False
    )
)

print()
print(
    f"Expansion rate: {top5_rate * 100:.2f}%"
)

print(
    f"Average retrieval depth: {average_k:.3f}"
)

print(
    f"Context reduction: {context_reduction * 100:.2f}%"
)

print(
    f"Latency reduction: {latency_reduction * 100:.2f}%"
)

print()
print(
    "Outputs saved to:",
    output_path
)

FINAL ADAPTIVE RAG RESULTS
                   Metric  Fixed Dense@5  Adaptive 3→5
              Exact Match       0.294000      0.305000
                       F1       0.437735      0.442568
       Avg Context Tokens     666.994000    412.025000
     Avg Generated Tokens       8.232000      7.774000
      Avg Retrieval Depth       5.000000      3.200000
   Avg Total Latency (ms)     504.022069    415.475997
Median Total Latency (ms)     381.043305    294.863834
   P95 Total Latency (ms)    1329.130250   1236.650165
         Top-3 Route Rate       0.000000      0.900000
     Top-5 Expansion Rate       1.000000      0.100000
  Context Token Reduction       0.000000      0.382266
        Latency Reduction       0.000000      0.175679

Expansion rate: 10.00%
Average retrieval depth: 3.200
Context reduction: 38.23%
Latency reduction: 17.57%

Outputs saved to: /kaggle/working/final_results_v2
